# Step 05 — Traffic Quality Gate & Corridor Probe Planning


In [1]:
import sys
import copy
import pickle
from datetime import datetime, timezone
from pathlib import Path

import networkx as nx
import pandas as pd
from IPython.display import display

import routing
import routing.traffic_pipeline as tp
import routing.traffic_quality as tq

from routing.config import ORIGIN, ORIGINS, DESTINATION, RESULTS_DIR
from routing.map_loader import load_road_graph
from routing.map_view import interactive_map
from routing.traffic_api import TomTomClient
from routing.traffic_pipeline import (
    collect_traffic_snapshot,
    make_routing_graph,
    save_summary,
)
from routing.traffic_quality import (
    plan_corridor_probes,
    audit_traffic_snapshot,
)
from routing.probe_history import (
    load_probe_history,
    update_probe_history,
    save_probe_history,
)

print("Python:", sys.version)
print("Kernel:", sys.executable)
print("Routing package:", Path(routing.__file__).resolve())
print("Traffic pipeline:", Path(tp.__file__).resolve())
print("Traffic quality:", Path(tq.__file__).resolve())


Python: 3.14.8 (main, Sep 30 2026, 17:55:09) [Clang 21.0.0 (clang-2100.3.34.2)]
Kernel: /Users/mojtabashabani/wiser/mary/.venv/bin/python
Routing package: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/src/routing/__init__.py
Traffic pipeline: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/src/routing/traffic_pipeline.py
Traffic quality: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/src/routing/traffic_quality.py


In [2]:
# Confirm that the expected package versions are loaded.
import inspect

print("plan_corridor_probes:", inspect.signature(plan_corridor_probes))
print("collect_traffic_snapshot:", inspect.signature(collect_traffic_snapshot))
print("update_probe_history:", inspect.signature(update_probe_history))

assert "history" in inspect.signature(plan_corridor_probes).parameters
assert "selected_edge_ids" in inspect.signature(collect_traffic_snapshot).parameters

print("PASS: Expected routing functions are loaded")


plan_corridor_probes: (graph: 'nx.MultiDiGraph', *, max_probes: 'int' = 12, origins: 'Mapping[str, tuple[float, float]] | None' = None, destination: 'tuple[float, float]' = (51.499, -0.1742), corridor_width_m: 'float' = 350.0, min_length_m: 'float' = 35.0, min_spacing_m: 'float' = 65.0, history: 'Mapping | None' = None) -> 'list[dict]'
collect_traffic_snapshot: (osm_graph: 'nx.MultiDiGraph', client: 'TomTomClient | None', *, max_requests: 'int' = 60, pause_s: 'float' = 0.1, confidence_threshold: 'float' = 0.65, selected_edge_ids: 'Iterable[tuple] | None' = None) -> 'tuple[nx.MultiDiGraph, dict]'
update_probe_history: (snapshot: 'nx.MultiDiGraph', summary: 'dict', history: 'dict | None' = None) -> 'dict'
PASS: Expected routing functions are loaded


## 1 — Load the OSM road graph


In [3]:
RADIUS_M = 2000

G = load_road_graph(radius_m=RADIUS_M)

print("Graph:", type(G).__name__)
print("Nodes:", G.number_of_nodes())
print("Directed edges:", G.number_of_edges())

assert isinstance(G, nx.MultiDiGraph)
assert G.is_directed()

print("PASS: OSM graph loaded")


Graph: MultiDiGraph
Nodes: 2569
Directed edges: 5850
PASS: OSM graph loaded


## 2 — Plan probes (no API request in this cell)


In [4]:
REQUEST_LIMIT = 6

history = load_probe_history()

plan = plan_corridor_probes(
    G,
    max_probes=REQUEST_LIMIT,
    origins=ORIGINS,
    destination=DESTINATION,
    corridor_width_m=350,
    min_length_m=35,
    min_spacing_m=65,
    history=history,
)

if not plan:
    raise RuntimeError(
        "No valid probes were planned. Check the OSM graph and corridor settings."
    )

plan_df = pd.DataFrame(plan)

print("Requested maximum probes:", REQUEST_LIMIT)
print("Planned probes:", len(plan))
print("\nRoad-class distribution:")
display(plan_df["road_class"].value_counts().rename("count").to_frame())
display(plan_df)


Requested maximum probes: 6
Planned probes: 6

Road-class distribution:


,count
road_class,
trunk,4
primary,2


,edge_id,latitude,longitude,length_m,road_class,corridor,corridor_offset_m
0,"(265565434, 109870, 0)",51.495027,-0.182279,72.25,trunk,Gloucester Road,34.0
1,"(20947178, 1691189798, 0)",51.501582,-0.177494,89.35,primary,Royal Albert Hall,77.2
2,"(2360746811, 6280192964, 0)",51.495728,-0.174309,107.16,trunk,South Kensington,7.6
3,"(265565438, 265565434, 0)",51.495122,-0.181017,103.78,trunk,Gloucester Road,76.0
4,"(385312258, 20947178, 0)",51.501568,-0.178813,93.45,primary,Royal Albert Hall,107.2
5,"(6280192964, 3818359076, 0)",51.495869,-0.172541,140.77,trunk,South Kensington,115.2


In [5]:
import json

USE_ROUTE_FOCUSED_PROBES = False
route_probe_path = RESULTS_DIR / "route_probe_candidates.json"

if USE_ROUTE_FOCUSED_PROBES:
    if not route_probe_path.exists():
        raise FileNotFoundError(
            f"{route_probe_path} not found. Run Notebook 06 first, or set "
            "USE_ROUTE_FOCUSED_PROBES = False."
        )

    selected_probe_records = json.loads(
        route_probe_path.read_text(encoding="utf-8")
    )

    if not selected_probe_records:
        raise ValueError(
            "The route-focused probe file is empty. No missing A* route edges were exported."
        )

    selected_probe_records = [
        {**item, "edge_id": tuple(item["edge_id"])}
        for item in selected_probe_records
    ]
    print("Using route-focused A* probe candidates:", len(selected_probe_records))

else:
    selected_probe_records = plan
    print("Using corridor-planned probes:", len(selected_probe_records))

selected_edge_ids = [
    tuple(item["edge_id"])
    for item in selected_probe_records
]

missing_edges = [
    edge_id for edge_id in selected_edge_ids
    if not G.has_edge(*edge_id)
]

print("Selected probes:", len(selected_edge_ids))
print("Unique directed edges:", len(set(selected_edge_ids)))
print("Missing edges:", missing_edges)

assert len(selected_edge_ids) == len(set(selected_edge_ids))
assert not missing_edges

selected_probe_df = pd.DataFrame(selected_probe_records)
display(selected_probe_df)

print("PASS: Every selected probe is a unique directed OSM edge")


Using corridor-planned probes: 6
Selected probes: 6
Unique directed edges: 6
Missing edges: []


,edge_id,latitude,longitude,length_m,road_class,corridor,corridor_offset_m
0,"(265565434, 109870, 0)",51.495027,-0.182279,72.25,trunk,Gloucester Road,34.0
1,"(20947178, 1691189798, 0)",51.501582,-0.177494,89.35,primary,Royal Albert Hall,77.2
2,"(2360746811, 6280192964, 0)",51.495728,-0.174309,107.16,trunk,South Kensington,7.6
3,"(265565438, 265565434, 0)",51.495122,-0.181017,103.78,trunk,Gloucester Road,76.0
4,"(385312258, 20947178, 0)",51.501568,-0.178813,93.45,primary,Royal Albert Hall,107.2
5,"(6280192964, 3818359076, 0)",51.495869,-0.172541,140.77,trunk,South Kensington,115.2


PASS: Every selected probe is a unique directed OSM edge


## 3 — Safe snapshot loading and merging



In [6]:
snapshot_path = RESULTS_DIR / "traffic_snapshot.pkl"

_TRAFFIC_VALUE_FIELDS = {
    "current_speed_kph",
    "free_flow_speed_kph",
    "travel_time_s",
    "free_flow_time_s",
    "time_is_imputed",
    "is_closed",
}

def _is_traffic_attribute(name):
    return name.startswith("traffic_") or name in _TRAFFIC_VALUE_FIELDS


def _assert_same_topology(base_graph, candidate_graph, label):
    if not isinstance(candidate_graph, nx.MultiDiGraph):
        raise TypeError(f"{label} must be a networkx.MultiDiGraph")

    if set(base_graph.nodes) != set(candidate_graph.nodes):
        raise ValueError(f"{label} node set differs from the current OSM graph")

    if set(base_graph.edges(keys=True)) != set(candidate_graph.edges(keys=True)):
        raise ValueError(f"{label} directed-edge set differs from the current OSM graph")


def merge_traffic_snapshots(base_graph, previous_snapshot, fresh_snapshot, checked_at_utc):
    """Merge the latest assessed edge attributes without discarding older unselected observations."""
    _assert_same_topology(base_graph, fresh_snapshot, "fresh_snapshot")

    if previous_snapshot is None:
        merged = base_graph.copy()
    else:
        _assert_same_topology(base_graph, previous_snapshot, "previous_snapshot")
        merged = previous_snapshot.copy()

    refreshed_edges = 0

    for u, v, key, fresh_attrs in fresh_snapshot.edges(keys=True, data=True):
        was_assessed = (
            bool(fresh_attrs.get("traffic_sampled"))
            or bool(fresh_attrs.get("traffic_directly_queried"))
            or fresh_attrs.get("traffic_match_source") == "reverse_inferred"
        )

        if not was_assessed:
            continue

        target_attrs = merged[u][v][key]

        # Remove old traffic fields for this edge, then copy the newest attempt.
        for attr_name in list(target_attrs):
            if _is_traffic_attribute(attr_name):
                target_attrs.pop(attr_name, None)

        for attr_name, value in fresh_attrs.items():
            if _is_traffic_attribute(attr_name):
                target_attrs[attr_name] = copy.deepcopy(value)

        target_attrs["traffic_last_checked_at_utc"] = checked_at_utc

        if (
            fresh_attrs.get("traffic_matched") is True
            and fresh_attrs.get("time_is_imputed") is False
            and fresh_attrs.get("traffic_status") in {"red", "yellow", "clear", "closed"}
        ):
            target_attrs["traffic_observed_at_utc"] = checked_at_utc

        refreshed_edges += 1

    merged.graph.update(copy.deepcopy(fresh_snapshot.graph))
    merged.graph["traffic_last_merge_refreshed_edges"] = refreshed_edges

    return merged, refreshed_edges


def count_usable_observations(graph):
    usable = 0
    sampled = 0

    for _, _, _, attrs in graph.edges(keys=True, data=True):
        if attrs.get("traffic_sampled"):
            sampled += 1

        if (
            attrs.get("traffic_matched") is True
            and attrs.get("time_is_imputed") is False
            and attrs.get("traffic_status") in {"red", "yellow", "clear", "closed"}
        ):
            usable += 1

    total = graph.number_of_edges()
    coverage = 100.0 * usable / total if total else 0.0

    return {
        "usable_observed_edges": usable,
        "sampled_directed_edges": sampled,
        "usable_coverage_percent": round(coverage, 3),
    }


previous_bundle = None
previous_snapshot = None

if snapshot_path.exists():
    with snapshot_path.open("rb") as f:
        previous_bundle = pickle.load(f)

    previous_snapshot = previous_bundle.get("snapshot")

    if previous_snapshot is not None:
        _assert_same_topology(G, previous_snapshot, "saved snapshot")
        print("Previous snapshot found:", snapshot_path)
else:
    print("No previous traffic snapshot found.")


Previous snapshot found: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/traffic_snapshot.pkl


## 4 — Collect traffic or load the last saved snapshot


In [7]:
RUN_LIVE_TRAFFIC = False  # Change to True only when you intend to call TomTom.

fresh_snapshot = None
batch_summary = None
refreshed_edges = 0

if RUN_LIVE_TRAFFIC:
    with TomTomClient() as client:
        fresh_snapshot, batch_summary = collect_traffic_snapshot(
            osm_graph=G,
            client=client,
            max_requests=len(selected_edge_ids),
            selected_edge_ids=selected_edge_ids,
            pause_s=0.5,
        )

    checked_at = (
        batch_summary.get("finished_at_utc")
        or datetime.now(timezone.utc).isoformat()
    )

    snapshot, refreshed_edges = merge_traffic_snapshots(
        G,
        previous_snapshot,
        fresh_snapshot,
        checked_at,
    )

    summary = dict(batch_summary)
    cumulative = count_usable_observations(snapshot)

    summary["merged_refreshed_edges"] = refreshed_edges
    summary["cumulative_usable_observed_edges"] = cumulative["usable_observed_edges"]
    summary["cumulative_sampled_directed_edges"] = cumulative["sampled_directed_edges"]
    summary["cumulative_usable_coverage_percent"] = cumulative["usable_coverage_percent"]

    print("Live collection completed.")
    print("Requests made:", summary.get("requests_made"))
    print("Newly refreshed directed edges:", refreshed_edges)

else:
    if previous_bundle is None or previous_snapshot is None:
        raise FileNotFoundError(
            "No saved traffic_snapshot.pkl exists. Set RUN_LIVE_TRAFFIC = True "
            "and rerun this cell to create the first snapshot."
        )

    snapshot = previous_snapshot
    summary = previous_bundle.get("summary", {})
    print("Loaded the previously saved traffic snapshot; no API request was made.")

_assert_same_topology(G, snapshot, "working snapshot")

print("Snapshot nodes:", snapshot.number_of_nodes())
print("Snapshot directed edges:", snapshot.number_of_edges())
print("Cumulative observation counts:", count_usable_observations(snapshot))


Loaded the previously saved traffic snapshot; no API request was made.
Snapshot nodes: 2569
Snapshot directed edges: 5850
Cumulative observation counts: {'usable_observed_edges': 6, 'sampled_directed_edges': 8, 'usable_coverage_percent': 0.103}


In [8]:
# Show the current planned edges against the working snapshot.
probe_rows = []

for item in selected_probe_records:
    edge_id = tuple(item["edge_id"])
    u, v, key = edge_id
    attrs = snapshot[u][v][key]

    probe_rows.append({
        "edge_id": edge_id,
        "road_class": item["road_class"],
        "corridor": item.get("corridor"),
        "sampled": attrs.get("traffic_sampled", False),
        "matched": attrs.get("traffic_matched", False),
        "match_source": attrs.get("traffic_match_source"),
        "match_reason": attrs.get("traffic_match_reason"),
        "traffic_status": attrs.get("traffic_status", "missing"),
        "current_speed_kph": attrs.get("current_speed_kph"),
        "free_flow_speed_kph": attrs.get("free_flow_speed_kph"),
        "confidence": attrs.get("traffic_confidence"),
        "time_is_imputed": attrs.get("time_is_imputed", True),
    })

probe_results_df = pd.DataFrame(probe_rows)
display(probe_results_df)


,edge_id,road_class,corridor,sampled,matched,match_source,match_reason,traffic_status,current_speed_kph,free_flow_speed_kph,confidence,time_is_imputed
0,"(265565434, 109870, 0)",trunk,Gloucester Road,False,False,NaN,NaN,missing,NaN,NaN,NaN,True
1,"(20947178, 1691189798, 0)",primary,Royal Albert Hall,True,True,reverse_inferred,reverse_candidate_match,yellow,18.0,26.0,1.0,False
2,"(2360746811, 6280192964, 0)",trunk,South Kensington,False,False,NaN,NaN,missing,NaN,NaN,NaN,True
3,"(265565438, 265565434, 0)",trunk,Gloucester Road,False,False,NaN,NaN,missing,NaN,NaN,NaN,True
4,"(385312258, 20947178, 0)",primary,Royal Albert Hall,True,True,reverse_inferred,reverse_candidate_match,yellow,18.0,26.0,1.0,False
5,"(6280192964, 3818359076, 0)",trunk,South Kensington,False,False,NaN,NaN,missing,NaN,NaN,NaN,True


## 5 — Update history and audit the merged snapshot


In [9]:
if RUN_LIVE_TRAFFIC:
    # Update history from this batch only, not from the cumulative merged snapshot.
    history = update_probe_history(
        fresh_snapshot,
        batch_summary,
        history=history,
    )

    history_path = save_probe_history(history)
    save_summary(summary)

    print("Probe history saved:", history_path)
else:
    print("History was not changed because no new API collection was run.")

audit = audit_traffic_snapshot(
    G,
    snapshot,
    summary=summary,
)

print("========== TRAFFIC SNAPSHOT AUDIT ==========")
for key, value in audit.items():
    print(f"{key}: {value}")

assert audit["graph_identity_ok"], "Snapshot topology/length audit failed"
assert audit["costs_ok"], "Snapshot contains invalid routing costs"
assert audit["algorithm_smoke_test_ready"], "Snapshot is not structurally ready"

print("PASS: Snapshot passed structural audit")


History was not changed because no new API collection was run.
========== TRAFFIC SNAPSHOT AUDIT ==========
graph_identity_ok: True
costs_ok: True
base_nodes: 2569
snapshot_nodes: 2569
total_directed_edges: 5850
sampled_directed_edges: 8
directly_queried_directed_edges: 6
matched_edges: 6
observed_edges: 6
direct_observed_edges: 4
reverse_inferred_observed_edges: 2
missing_edges: 5844
imputed_edges: 5844
invalid_status_edges: 0
observed_coverage_percent: 0.103
direct_observed_coverage_percent: 0.068
reverse_observed_coverage_percent: 0.034
algorithm_smoke_test_ready: True
final_traffic_comparison_ready: False
warnings: ['Observed traffic coverage is sparse; most travel times are not measured', 'TomTom geometry point order is not authoritative legal direction proof']
PASS: Snapshot passed structural audit


## 6 — Build the routing graphs for Notebook 06


In [10]:
free_flow_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="free_flow",
)

measured_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="measured",
)

measured_distance = make_routing_graph(
    snapshot,
    objective="distance",
    scenario="measured",
)

print("Free-flow graph edges:", free_flow_time.number_of_edges())
print("Measured-time graph edges:", measured_time.number_of_edges())
print("Measured-distance graph edges:", measured_distance.number_of_edges())

print("PASS: Routing graphs prepared")


Free-flow graph edges: 5850
Measured-time graph edges: 5850
Measured-distance graph edges: 5850
PASS: Routing graphs prepared


## 7 — Save the exact snapshot used by Notebook 06



In [11]:
if RUN_LIVE_TRAFFIC:
    previous_runs = []
    if previous_bundle is not None:
        previous_runs = list(previous_bundle.get("collection_runs", []))

    previous_runs.append({
        "finished_at_utc": summary.get("finished_at_utc"),
        "requests_made": summary.get("requests_made"),
        "request_errors": summary.get("request_errors"),
        "newly_refreshed_edges": refreshed_edges,
        "cumulative_usable_observed_edges": summary.get(
            "cumulative_usable_observed_edges"
        ),
    })

    traffic_bundle = {
        "snapshot": snapshot,
        "summary": summary,
        "plan": selected_probe_records,
        "corridor_plan": plan,
        "selected_edge_ids": selected_edge_ids,
        "collection_runs": previous_runs,
        "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    with snapshot_path.open("wb") as f:
        pickle.dump(
            traffic_bundle,
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )

    print("Saved the merged snapshot:", snapshot_path)
    print("Saved observed edges:", count_usable_observations(snapshot)["usable_observed_edges"])
else:
    print("No file was rewritten because RUN_LIVE_TRAFFIC is False.")
    print("Notebook 06 will read the existing snapshot:", snapshot_path)


No file was rewritten because RUN_LIVE_TRAFFIC is False.
Notebook 06 will read the existing snapshot: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/traffic_snapshot.pkl


## 8 — Optional interactive map check


In [12]:
traffic_widget, ui_state = interactive_map(
    G,
    origin=ORIGIN,
    destination=DESTINATION,
    traffic_graph=snapshot,
)

display(traffic_widget)


In [14]:
print("Origin node:", ui_state["origin_node"])
print("Destination node:", ui_state["destination_node"])
print("Last double-click:", ui_state["last_click_latlon"])

assert ui_state["origin_node"] in G
assert ui_state["destination_node"] in G

if ui_state["last_click_latlon"] is None:
    print("Map loaded correctly. No new double-click was made in this session.")
else:
    print("PASS: Interactive map received a double-click.")


Origin node: 27481160
Destination node: 26389375
Last double-click: (51.49879538421187, -0.18141154960951814)
PASS: Interactive map received a double-click.
